In [ ]:
import destinelab as deauth
import requests
import json
import os
import zipfile
import datetime
import shutil
from getpass import getpass

In [ ]:
DESP_USERNAME = input("Please input your DESP username or email: ")
DESP_PASSWORD = getpass("Please input your DESP password: ")

auth = deauth.AuthHandler(DESP_USERNAME, DESP_PASSWORD)
access_token = auth.get_token()
if access_token is not None:
    print("DEDL/DESP Access Token Obtained Successfully")
else:
    print("Failed to Obtain DEDL/DESP Access Token")

auth_headers = {"Authorization": f"Bearer {access_token}"}

In [ ]:
HDA_STAC_ENDPOINT="https://hda.data.destination-earth.eu/stac/v2"
COLLECTION_ID = "EO.EUM.DAT.MTG.FCI-ACTIVE_FIRE-L2-V1"
response = requests.post(HDA_STAC_ENDPOINT+"/search", headers=auth_headers, json={
    "collections": [COLLECTION_ID],
    "datetime": "2025-08-15T15:00:00Z/2025-08-15T16:00:00Z"

})
from IPython.display import JSON

product = response.json()["features"][0]
JSON(product)
from tqdm import tqdm
import time

# Define a list of assets to download
assets = ["downloadLink"]

for asset in assets:
    download_url = product["assets"][asset]["href"]
    print(download_url)
    filename = asset
    print(filename)
    response = requests.get(download_url, headers=auth_headers)
    total_size = int(response.headers.get("content-length", 0))

    print(f"downloading {filename}")

    with tqdm(total=total_size, unit="B", unit_scale=True) as progress_bar:
        with open(filename, 'wb') as f:
            for data in response.iter_content(1024):
                progress_bar.update(len(data))
                f.write(data)

zf=zipfile.ZipFile(filename)
with zipfile.ZipFile(filename, 'r') as zip_ref:
    zip_ref.extractall('.')

import xarray as xr
import numpy as np
ds = xr.open_dataset('W_XX-EUMETSAT-Darmstadt,IMG+SAT,MTI1+FCI-2-FIR--FD------NC4E_C_EUMT_20250815150458_L2PF_OPE_20250815145000_20250815150000_N__C_0090_0000.nc').sel(number_of_rows = slice(4000,5000), number_of_columns = slice(2000,3000))



In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import ListedColormap
import matplotlib.patches as mpatches

# Sostituisci 'variable_name' con il nome della tua variabile
da = ds["fire_result"]

# Maschera i valori 4 (missing)
da = da.where(da != 4)

# Definisci una colormap discreta (4 classi)
colors = ["lightgray", "orange", "red", "darkred"]
cmap = ListedColormap(colors)

# Plot
plt.figure(figsize=(8, 6))
im = da.plot(cmap=cmap, add_colorbar=False)

# Crea legenda personalizzata (senza numeri)
labels = [
    "No fire",
    "Fire low confidence (20%<=fire_probability<40%)",
    "Fire medium confidence (40%<=fire_probability<80%)",
    "Fire high confidence (80%<=fire_probability)"
]

patches = [mpatches.Patch(color=colors[i], label=labels[i]) for i in range(4)]

plt.legend(handles=patches, loc="lower right", frameon=True)

plt.title("Fire classification")
plt.show()